# Aggregate (quartz) dissolution in a hydrated cement system

**Authors (original):** G. Dan Miron, Georg Kosakowski   **Refactor:** parameterized framework with extensible kinetics

## What this notebook does

Simulates kinetically controlled dissolution of quartz aggregate in a hydrated cement / mortar system over geological time scales, using xGEMS for the underlying multiphase equilibrium calculation. Quartz is constrained; the rest of the system equilibrates freely every timestep.

## How to use

**You only need to edit the CONFIG cell.** Everything else just runs.

The framework lives in `cement_dissolution.py`. To add a new material (a corroding metal, a degrading organic), subclass `KineticMaterial` there and add an instance to the `materials` list below — the simulation loop, output, and plotting all handle it automatically.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cement_dissolution import (
    # Simulation orchestrator
    CementDissolutionSimulation,
    # Config dataclasses
    TimeConfig, OutputConfig, CompositionOverride,
    # Kinetic materials
    QuartzTST, MetalCorrosion, OrganicDegradation,
    # Surface area models
    ConstantSurface, SpecificSurface, ShrinkingSphereSurface,
    # Coupling hooks
    FixedPHHook,
    # Helpers
    years_to_seconds, celsius_to_kelvin, ca_si_in_phase_factory,
)

plt.rcParams.update({'font.size': 13})

## ⚙️  Configuration   ← *this is the only cell you normally edit*

Five blocks below correspond to the five things you typically vary:

1. **System file & temperature** — which GEMS thermodynamic database, what T.
2. **Initial composition** — programmatic overrides of mineral amounts / bulk elements (no need to edit the .lst file).
3. **Kinetic materials** — the heart of the simulation. Add metals and organics here.
4. **Coupling hooks** — pH control, redox buffering, anything the engine can't set directly.
5. **Time stepping & outputs** — horizon, step size, what to record.

In [ ]:
# ============================================================
# 1) SYSTEM FILE & TEMPERATURE
# ============================================================
SYSTEM_FILE   = 'gems_files/mortar-dat.lst'
TEMPERATURE_K = celsius_to_kelvin(25.0)   # = 298.15 K

# ============================================================
# 2) INITIAL COMPOSITION OVERRIDE
#    Change the mortar composition without editing the .lst file.
#    Lower limits force a minimum amount; upper limits suppress a phase.
#    bulk_elements directly overwrites the engine's b vector.
# ============================================================
composition_override = CompositionOverride(
    species_lower_limits = {
        'Qtz': 26.679,                  # mol of quartz aggregate (original value)
    },
    species_upper_limits = {
        # 'OH-hydrotalcite': 1e-9,      # uncomment to suppress hydrotalcite
    },
    bulk_elements = {
        # 'CaO2': 5.2,                    
        # 'SiO2': 3.1,
        # 'Al2O3': 0.6,
        # cement clinkers ...
    },
)

# ============================================================
# 3) KINETIC MATERIALS
#    Add an instance per material. Each subclass of KineticMaterial
#    controls one xGEMS species. New materials require only that you
#    write a compute_rate(state, current_amount) method.
# ============================================================
materials = [
    QuartzTST(
        species_name      = 'Qtz',
        saturation_phase  = 'Quartz',
        surface           = SpecificSurface(9.065e-2),  # m^2/mol, ~1.5 mm grains
        # neutral mechanism (Palandri & Kharaka 2004)
        k_neutral = 6.4e-14, Ea_neutral = 77_000.0,
        # acid mechanism (off)
        k_acid = 0.0,        Ea_acid = 0.0,    n_acid = 0.0,
        # base mechanism
        k_base = 1.9e-10,    Ea_base = 80_000.0, n_base = 0.34,
    ),

    # --- Examples for future use (uncomment & adjust species names ----------
    # MetalCorrosion(
    #     species_name = 'Fe(metal)',
    #     surface      = ShrinkingSphereSurface(initial_area_m2=1.0),
    #     k_corr       = 1.0e-11,  # mol m^-2 s^-1, illustrative
    #     Ea           = 40_000.0,
    # ),
    # OrganicDegradation(
    #     species_name = 'Cellulose',
    #     k_decay      = 1.0e-11,  # 1/s
    #     Q10          = 2.0,
    # ),
]

# ============================================================
# 4) COUPLING HOOKS  (pre / post equilibration)
# ============================================================
hooks = [
    # FixedPHHook(target_pH=12.5, pH_handle='Cl'),  # stub — extend for real pH control
]

# ============================================================
# 5) TIME STEPPING & OUTPUTS
# ============================================================
time_config = TimeConfig(
    t_end_s       = years_to_seconds(5000.0),
    dt_initial_s  = years_to_seconds(1.0),
    accel_after_s = years_to_seconds(1000.0),
    accel_factor  = 1.01,
    dt_max_s      = None,    # set e.g. years_to_seconds(50) to cap dt
)

# Aqueous elements: total concentration in solution (mmol/L of aqueous phase)
# Solid phase groups: display-name -> list of GEMS phases that share that display name
# (collapses e.g. three ettringite variants into one stack)
output_config = OutputConfig(
    aqueous_elements = ['Ca', 'Si', 'Al', 'Mg', 'S', 'C'],
    aqueous_phase    = 'aq_gen',
    phase_groups = {
        'quartz':         ['Quartz'],
        'CSH':            ['CSHQ'],
        'ettringite':     ['ettringite', 'SO4_CO3_AFt', 'CO3_SO4_AFt'],
        'thaumasite':     ['thaumasite'],
        'monosulphate':   ['SO4_OH_AFm', 'OH_SO4_AFm'],
        'monocarbonate':  ['C4AcH11'],
        'hydrotalcite':   ['OH-hydrotalcite'],
        'Kuzel_s':        ['Kuzels'],
        'Friedel_s':      ['Friedels'],
        'straetlingite':  ['straetlingite'],
        'calcite':        ['Calcite'],
        'MSH':            ['MSH'],
        'gypsum':         ['Gypsum'],
        'portlandite':    ['Portlandite'],
        'zeolites':       ['Natrolite', 'ZeoliteX', 'ZeoliteY', 'ZeoliteP', 'Chabazite'],
        'hydrogarnet':    ['C3(AF)S0.84H', 'C3FS1.34H3.32'],
        'ferrihydrite':   ['Ferrihydrite-mc'],
        'Al(OH)3(mic)':   ['Al(OH)3mic'],
    },
    extras = {
        'Ca/Si in CSH': ca_si_in_phase_factory('CSHQ', 'Ca', 'Si'),
    },
)

## Run the simulation

In [ ]:
sim = CementDissolutionSimulation(
    system_file          = SYSTEM_FILE,
    materials            = materials,
    temperature_K        = TEMPERATURE_K,
    time_config          = time_config,
    output_config        = output_config,
    composition_override = composition_override,
    hooks                = hooks,
    verbose              = True,
)

results = sim.run()

## Inspect the results

Convert to pandas (if installed) for tabular inspection, or work with `results.aqueous_mmol_per_L`, `results.solids_cm3`, `results.rates_mol_per_s`, and `results.extras` directly as dicts of lists.

In [ ]:
try:
    props_df, aq_df, solids_df = results.to_dataframes()
    display(props_df.head())
    display(aq_df.head())
except ImportError:
    print('pandas not installed; results.<field> still works as dicts of lists.')
    print('first 5 timesteps (years):', results.time_years[:5])
    print('first 5 pH values:        ', results.pH[:5])

# Number of failed (NaN) timesteps, if any:
import math
nan_steps = sum(1 for x in results.pH if isinstance(x, float) and math.isnan(x))
print(f'\nfailed equilibrations: {nan_steps} / {len(results.time_years)} timesteps')

## Save results to disk

`results.save()` writes the same three tables you saw above plus a `metadata.json` sidecar with the simulation config. 
Default format is CSV (three files in a `results/<run_name>/` subdirectory, no extra dependencies). Pass `format='xlsx'` for a single workbook with one sheet per table (requires `pandas` + `openpyxl`).

Each call uses a timestamped `run_name` by default, so re-running the notebook keeps every run instead of overwriting. Pass an explicit `run_name='my_case'` and `overwrite=True` to control that.

In [ ]:
saved_path = results.save(
    output_dir = 'results',
    run_name   = None,            # default: 'run_YYYYMMDD_HHMMSS'
    format     = 'csv',           # or 'xlsx'
    overwrite  = False,
    metadata   = {
        # captured config — useful for sweeps / reproducibility
        'system_file':   SYSTEM_FILE,
        'temperature_K': TEMPERATURE_K,
        'temperature_C': TEMPERATURE_K - 273.15,
        't_end_years':   time_config.t_end_s / years_to_seconds(1.0),
        'dt_initial_years': time_config.dt_initial_s / years_to_seconds(1.0),
        'materials':     [type(m).__name__ for m in materials],
        'hooks':         [type(h).__name__ for h in hooks],
        'composition_override': {
            'species_lower_limits': dict(composition_override.species_lower_limits),
            'species_upper_limits': dict(composition_override.species_upper_limits),
            'bulk_elements':        dict(composition_override.bulk_elements),
        },
        'aqueous_elements': list(output_config.aqueous_elements),
        'phase_groups':     {k: list(v) for k, v in output_config.phase_groups.items()},
    },
)
print(f'Results written to: {saved_path}')

## Plot 1 — aqueous concentrations and pH

In [ ]:
fig, _ = results.plot_aqueous_and_pH(title='Quartz dissolution: aqueous + pH')
plt.show()

## Plot 2 — solid phase volume evolution

Stacked area of all solid phase groups, with Ca/Si in the CSH phase on a twin axis. Adjust `ylim` to zoom; set `log_x=True` for log time.

In [ ]:
fig, _ = results.plot_solid_volumes(
    title             = 'Solid phase volume evolution during quartz dissolution',
    secondary_series  = 'Ca/Si in CSH',
    secondary_label   = 'Ca/Si in CSH',
    ylim              = (600, 1000),
    log_x             = False,
)
plt.show()